In [1]:
import polars as pl

In [2]:
gldv_merged = pl.read_csv(
    "/mnt/yokoyamalab-nas/gldv2-full/csv/v2/merged_set_top_countries_caption_labels.csv"
)
gldv_merged.head()

id,landmark_id,latitude,longitude,wikimedia_url,geohack_url,country_code,country,region,subregion,src,pred_label,pred_score,caption
str,i64,f64,f64,str,str,str,str,str,str,str,str,f64,str
"""4b966789e0572456""",30319,52.226169,21.023186,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""PL""","""Poland""","""Europe""","""Eastern Europe""","""train""","""palace_castle""",0.9,"""Classical European building wi…"
"""16d8aa057cdd01b9""",25719,45.58359,9.27567,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""IT""","""Italy""","""Europe""","""Southern Europe""","""train""","""non_landmark""",0.9,"""Information board with text an…"
"""f1556e1eeeba213f""",36639,44.0446,-74.9325,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""US""","""United States""","""North America""","""Northern America""","""train""","""non_landmark""",0.9,"""A small green plant growing on…"
"""4072182eddd0100e""",2474,51.685278,-2.543611,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""GB""","""United Kingdom""","""Europe""","""Northern Europe""","""train""","""non_landmark""",0.9,"""A serene stream flowing throug…"
"""3cdc355c6232712f""",12877,48.862444,2.395889,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""FR""","""France""","""Europe""","""Western Europe""","""train""","""non_landmark""",0.9,"""A memorial wall with numerous …"


In [4]:
valid_df = gldv_merged.filter(pl.col("pred_label") != "non_landmark").with_columns(
    (pl.col("country") + "|" + pl.col("pred_label")).alias("stratify_key")
)
valid_df

id,landmark_id,latitude,longitude,wikimedia_url,geohack_url,country_code,country,region,subregion,src,pred_label,pred_score,caption,stratify_key
str,i64,f64,f64,str,str,str,str,str,str,str,str,f64,str,str
"""4b966789e0572456""",30319,52.226169,21.023186,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""PL""","""Poland""","""Europe""","""Eastern Europe""","""train""","""palace_castle""",0.9,"""Classical European building wi…","""Poland|palace_castle"""
"""202cd79556f30760""",104169,56.123889,-3.947778,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""GB""","""United Kingdom""","""Europe""","""Northern Europe""","""train""","""palace_castle""",0.9,"""Ancient stone castle on a hill…","""United Kingdom|palace_castle"""
"""e9028e1ef00e6cff""",19756,43.275738,11.990018,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""IT""","""Italy""","""Europe""","""Southern Europe""","""train""","""arch_gate""",0.9,"""A stone arch gate with dark wo…","""Italy|arch_gate"""
"""6e901d6b9d4da278""",101718,43.881986,11.097297,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""IT""","""Italy""","""Europe""","""Southern Europe""","""train""","""tower""",0.9,"""A tall, striped tower with a l…","""Italy|tower"""
"""cc6b89c09b881029""",147865,31.21899,121.464163,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""CN""","""China""","""Asia""","""Eastern Asia""","""train""","""arch_gate""",0.9,"""Black metal gate with a plaque…","""China|arch_gate"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""514bdb4e3aaf5fc0""",73880,48.021944,11.861944,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""DE""","""Germany""","""Europe""","""Western Europe""","""index""","""tower""",0.9,"""Gothic cathedral tower with in…","""Germany|tower"""
"""4f444b0bba8cdcc0""",31992,32.921,-117.253,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""US""","""United States""","""North America""","""Northern America""","""index""","""arch_gate""",0.9,"""Stone building with a sign rea…","""United States|arch_gate"""
"""be3420d695838ba3""",49644,45.466669,9.197553,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""IT""","""Italy""","""Europe""","""Southern Europe""","""index""","""palace_castle""",0.9,"""A grand neoclassical building …","""Italy|palace_castle"""


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(valid_df, test_size=0.2, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=0.1, random_state=42)

len(train_df), len(valid_df), len(test_df)

(601315, 66813, 167033)

In [8]:
train_df.head()

id,landmark_id,latitude,longitude,wikimedia_url,geohack_url,country_code,country,region,subregion,src,pred_label,pred_score,caption,stratify_key
str,i64,f64,f64,str,str,str,str,str,str,str,str,f64,str,str
"""5224830a19a3f31c""",67085,34.1211,133.001,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""JP""","""Japan""","""Asia""","""Eastern Asia""","""train""","""bridge""",0.9,"""A large suspension bridge with…","""Japan|bridge"""
"""b2d54da00a70bf24""",152227,53.006,7.192,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""NL""","""Netherlands""","""Europe""","""Western Europe""","""train""","""bridge""",0.9,"""A red wooden bridge over a pon…","""Netherlands|bridge"""
"""843af7cc5cf86f15""",64521,48.431806,0.090278,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""FR""","""France""","""Europe""","""Western Europe""","""train""","""arch_gate""",0.9,"""Classical French architecture …","""France|arch_gate"""
"""5c7b744683854d46""",58384,46.002,12.2867,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""IT""","""Italy""","""Europe""","""Southern Europe""","""index""","""palace_castle""",0.9,"""A medieval castle on a hilltop…","""Italy|palace_castle"""
"""a2510e80d6442d4f""",11054,40.42189,-3.682189,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…","""ES""","""Spain""","""Europe""","""Southern Europe""","""train""","""monument_statue""",0.9,"""A dynamic statue of a winged f…","""Spain|monument_statue"""


In [9]:
train_df.write_csv("/mnt/yokoyamalab-nas/gldv2-full/csv/v2/train.csv")
valid_df.write_csv("/mnt/yokoyamalab-nas/gldv2-full/csv/v2/valid.csv")
test_df.write_csv("/mnt/yokoyamalab-nas/gldv2-full/csv/v2/test.csv")